# Imaris + Cellpose -> Vizgen-Compat
This notebook convert the Imaris puncta detection output and Cellpose segmentation into Vizgen-compatible formats for downstream analysis

In [1]:
import os
import polars as pl
import plotly.graph_objects as go
import tifffile
from PIL import Image as PILImage
import typing

In [2]:
class ImsAxesInfo(typing.TypedDict):
    pixels: int | None
    microns_per_pixel: float | None
class ImsImageInfo(typing.TypedDict):
    X: ImsAxesInfo
    Y: ImsAxesInfo

---

## Paths Setup
Update the path to relevant files and directories here. The output directory will be created if it doesn't exist.

In [3]:
# The main data directory where raw data is saved
DATA_DIR = "D:/JOBS/WashU_Neuroscience/RNAScope/RNAScope_DATA/TEST_ACxDev1_CBA-CAJ_P18_Male_LACx_ONLY/data"
IMG_DIR = os.path.join(DATA_DIR, "images")

# Imaris' puncta detection output
SPOT_CSV = os.path.join(DATA_DIR, "All shank3 puncta.csv")

# Open the Imaris .ims file in Imaris Viewer, type Ctrl+I to open the Image Properties Panel
# then, copy the X and Y axes information into the ImsImageInfo below
# Behavior:
# - microns_per_pixel=None -> inherit from RAW_CZI metadata
# - pixels=None -> keep current behavior (infer using 'Distance to Image Border XY' in SPOT_CSV)
# - pixels is set -> compare IMS vs CZI axis metadata to derive corrected spot_scale
IMARIS_IMS_FILE_IMAGE_INFO = ImsImageInfo(
    X = ImsAxesInfo(
        pixels = 37824  , # If None, infer from 'Distance to Image Border XY' in SPOT_CSV, otherwise use this IMS value
        microns_per_pixel = None # if None, inherit from RAW_CZI metadata; set explicitly only when needed
    ),
    Y = ImsAxesInfo(
        pixels = 11454  , # If None, infer from 'Distance to Image Border XY' in SPOT_CSV, otherwise use this IMS value
        microns_per_pixel = None # if None, inherit from RAW_CZI metadata; set explicitly only when needed
    )
)

# Raw CZI of the data region
RAW_CZI = os.path.join(IMG_DIR, "P18LACx Layer1-6 40x.czi")

# Processed TIFF used for Cellpose segmentation and the resulting Cellpose segmentation mask
PROCESSED_TIFF = os.path.join(IMG_DIR, "P18LACx Layer1-6 40x-Airyscan Processing-01-Create Image Subset-02_c1-2.tif")
CELLPOSE_NPY = os.path.join(IMG_DIR, "P18LACx Layer1-6 40x-Airyscan Processing-01-Create Image Subset-02_c1-2_seg.npy")

In [4]:
# Keep as DATA_DIR for similar structure to MERSCOPE's output_data directory
# Otherwise, change the output location for preprocessing here
OUTPUT_DIR = os.path.join(DATA_DIR)

---

## Load + Validate Cellpose Segmentation

In [5]:
if not os.path.exists(OUTPUT_DIR):
	os.makedirs(OUTPUT_DIR)

In [6]:
from czifile import CziFile
from pint import UnitRegistry
ureg = UnitRegistry()

axes_to_log = ['X', 'Y', 'Z']
size_metadata = {ax: {} for ax in axes_to_log}
with CziFile(RAW_CZI) as czi:
	img = czi.scenes[0]
	for ax in axes_to_log:
		# Get the index of this axis in the CZI file
		ax_index = img.dims.index(ax)
		# Get physical size in micron/pixel
		scale = img.coord_scales[ax] * ureg(img.coord_units[ax])
		micron_per_pixel = scale.to("micron").magnitude
		# Get the FOV size in both pixels and microns
		size_pixels = img.shape[ax_index]
		size_microns = size_pixels * micron_per_pixel
		size_metadata[ax] = {'pixels': size_pixels, 'microns': size_microns, 'micron_per_pixel': micron_per_pixel}

print("Extracted size metadata from CZI file:")
size_metadata

Extracted size metadata from CZI file:


{'X': {'pixels': 37273,
  'microns': 1930.37467590332,
  'micron_per_pixel': 0.05179016113281249},
 'Y': {'pixels': 11469,
  'microns': 593.9813580322265,
  'micron_per_pixel': 0.05179016113281249},
 'Z': {'pixels': 7, 'microns': 10.5, 'micron_per_pixel': 1.5}}

In [7]:
import numpy as np

cellpose_output = np.load(CELLPOSE_NPY, allow_pickle=True)
print(cellpose_output.item().keys())
cp_outlines = cellpose_output.item().get("outlines", [])
cp_masks = cellpose_output.item().get("masks", [])
print(f"outlines shape: {cp_outlines.shape}, mask shape: {cp_masks.shape}")

dict_keys(['outlines', 'colors', 'masks', 'filename', 'flows', 'ismanual', 'manual_changes', 'model_path', 'flow_threshold', 'cellprob_threshold', 'normalize_params', 'restore', 'ratio', 'diameter'])
outlines shape: (2864, 9315), mask shape: (2864, 9315)


In [8]:
# Convert the processed_tiff/cp_mask dimensions to match the physical size of the original CZI
cp_size = cp_masks.shape # NOTE!! cellpose and tiff dimensions are (Y, X), flip to match CZI metadata
cp_size = (cp_size[1], cp_size[0])  # (X, Y)
# Calculate scaling factors
cp_scale_x = size_metadata['X']['microns'] / cp_size[0]
cp_scale_y = size_metadata['Y']['microns'] / cp_size[1]
print(f"Scaling factors for cellpose shape - X: {cp_scale_x:.4f} µm/pixel, Y: {cp_scale_y:.4f} µm/pixel")

Scaling factors for cellpose shape - X: 0.2072 µm/pixel, Y: 0.2074 µm/pixel


In [9]:
tif_img = tifffile.imread(PROCESSED_TIFF)

display_img = tif_img.copy()
display_img[cp_outlines > 0] = [255, 255, 255]

H, W = display_img.shape[:2]

# Build micron-spaced ticks from pixel positions
def _nice_interval(extent, n_ticks=10):
    raw = extent / n_ticks
    mag = 10 ** np.floor(np.log10(raw))
    n = raw / mag
    nice = 1 if n < 1.5 else 2 if n < 3 else 5 if n < 7 else 10
    return nice * mag

x_um = W * cp_scale_x
y_um = H * cp_scale_y

x_um_ticks = np.arange(0, x_um, _nice_interval(x_um))
y_um_ticks = np.arange(0, y_um, _nice_interval(y_um))

# Downsample with Lanczos anti-aliasing so Kaleido receives a small image without aliasing artefacts.
MAX_EXPORT_PX = 6000
scale = min(1.0, MAX_EXPORT_PX / max(W, H))
eW, eH = int(W * scale), int(H * scale)
export_img = np.array(PILImage.fromarray(display_img).resize((eW, eH), PILImage.Resampling.LANCZOS))
print(f"Original: {W}x{H}, export: {eW}x{eH} (scale={scale:.3f})")

# Dynamically scale font and line sizes relative to a 700px reference height (where ~12pt font looks good)
_REF_H = 700
title_font = max(12, round(14 * eH / _REF_H))
tick_label_font = max(10, round(12 * eH / _REF_H))

export_fig = go.Figure(go.Image(z=export_img))
export_fig.update_layout(
    paper_bgcolor="black",
    plot_bgcolor="black",
    margin=dict(l=60, r=20, t=20, b=60),
    width=eW,
    height=eH,
    xaxis=dict(
        tickmode="array",
        tickvals=(x_um_ticks / cp_scale_x * scale).tolist(),
        ticktext=[f"{v:.0f}" for v in x_um_ticks],
        title=dict(text="µm", font=dict(color="white", size=title_font)),
        showgrid=True,
        gridcolor="rgba(255,255,255,0.5)",
        zeroline=False,
        tickfont=dict(color="white", size=tick_label_font),
    ),
    yaxis=dict(
        tickmode="array",
        tickvals=(y_um_ticks / cp_scale_y * scale).tolist(),
        ticktext=[f"{v:.0f}" for v in y_um_ticks],
        title=dict(text="µm", font=dict(color="white", size=title_font)),
        showgrid=True,
        gridcolor="rgba(255,255,255,0.5)",
        zeroline=False,
        tickfont=dict(color="white", size=tick_label_font),
    ),
)

imgoutpath = os.path.normpath(os.path.join(OUTPUT_DIR, "cellpose_overlay_qc.webp"))
export_fig.write_image(imgoutpath, format="webp")
print(f"Saved to: {imgoutpath}")


Original: 9315x2864, export: 6000x1844 (scale=0.644)
Saved to: D:\JOBS\WashU_Neuroscience\RNAScope\RNAScope_DATA\TEST_ACxDev1_CBA-CAJ_P18_Male_LACx_ONLY\data\cellpose_overlay_qc.webp


## Processing Imaris's puncta `csv`

In [10]:
floatcolskeywords = ['area', 'ellipticity', 'ellipsoid', 'distance', 'length', 'size', 'sphericity', 'centroid', 'center', 'position', 'intensity', 'time', 'volume']
intcolskeywords = ['count']

# Polars auto skip leading empty rows it seems
_tmp = pl.scan_csv(SPOT_CSV, has_header=True, skip_rows=2, n_rows=0)
_colnames = _tmp.collect_schema().names()
_f64_overrides = {col: pl.Float64 for col in _colnames if any(keyword in col.lower() for keyword in floatcolskeywords)}
_i64_overrides = {col: pl.Int64 for col in _colnames if any(keyword in col.lower() for keyword in intcolskeywords)}

spot_stats = pl.scan_csv(SPOT_CSV, has_header=True, skip_rows=2,
	schema_overrides={**_f64_overrides, **_i64_overrides}
)
# Remove columns without header
spot_stats = spot_stats.select(pl.exclude(""))

spot_stats_df = spot_stats.collect()
columns = spot_stats_df.columns

# Get the min and max of cols starts with "Position X", "Position Y", "Position Z"
axes_colnames = {ax: next(c for c in columns if c.startswith(f"Position {ax}")) for ax in ['X', 'Y', 'Z']}
axes_colnames

{'X': 'Position X [µm]', 'Y': 'Position Y [µm]', 'Z': 'Position Z [µm]'}

In [11]:
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks

n_slices = size_metadata['Z']['pixels']
max_z = size_metadata['Z']['microns']
mpp = size_metadata['Z']['micron_per_pixel']
z_col = axes_colnames['Z']
z_vals = spot_stats_df[z_col]
z = z_vals.drop_nulls().to_numpy()

# Smooth a fine histogram to find natural valleys between Z-slice clusters
counts, edges = np.histogram(z, bins=500)
centers = (edges[:-1] + edges[1:]) / 2
smoothed = gaussian_filter1d(counts.astype(float), sigma=5)

# Valleys = local minima; keep the n_slices-1 deepest ones as bin boundaries
valley_idx, _ = find_peaks(-smoothed, prominence=smoothed.max() * 0.05)
deepest = valley_idx[np.argsort(smoothed[valley_idx])[:n_slices - 1]]
deepest = np.sort(deepest)
bins = np.concatenate([[z.min()], centers[deepest], [z.max()]])

print(f"Found {len(bins) - 1} bins with edges at: {np.round(bins, 3)}")

z_tick_vals = np.arange(0, max_z + mpp, mpp)
z_tick_labels = [f"{v:.1f}" for v in z_tick_vals]

fig = go.Figure()
fig.add_trace(go.Histogram(x=z_vals, nbinsx=100, name="Spots"))
fig.add_trace(go.Scatter(x=centers, y=smoothed, mode="lines", name="Smoothed density",
	line=dict(color="orange", width=2), yaxis="y"))
for edge in bins[1:-1]:
    fig.add_vline(x=edge, line_width=1, line_dash="dash", line_color="red", opacity=0.7)

fig.update_layout(
    title=f"Distribution of Spot Position Z ({n_slices} slices — valley-based bins)",
	paper_bgcolor="white",
	plot_bgcolor="rgba(255,255,255,0)",
    xaxis=dict(
        title="Position Z (µm)",
        range=[0, max_z],
        tickmode="array",
        tickvals=z_tick_vals.tolist(),
        ticktext=z_tick_labels,
        showgrid=False,
        gridcolor="lightgray",
    ),
    yaxis=dict(
        title="Count",
        range=[0, None],
        showgrid=True,
        gridcolor="lightgray",
    ),
    bargap=0.1,
    legend=dict(x=0.01, y=0.99),
)
fig.show()

imgoutpath = os.path.normpath(os.path.join(OUTPUT_DIR, "z_position_distribution.webp"))
fig.write_image(imgoutpath, format="webp")
print(f"Saved to: {imgoutpath}")

Found 7 bins with edges at: [0.75  1.137 2.901 4.449 5.943 7.455 9.363 9.75 ]


Saved to: D:\JOBS\WashU_Neuroscience\RNAScope\RNAScope_DATA\TEST_ACxDev1_CBA-CAJ_P18_Male_LACx_ONLY\data\z_position_distribution.webp


In [12]:
# Imaris puncta detection yields sub-layer Z values (probably somehow 3D inferred from the raw Z stack?)
# Re-bin the Z layers so that it's comparable to MERFISH data

# Assign each spot to a Z layer (0-indexed) using the valley-based bin edges
z_arr = spot_stats_df[z_col].to_numpy()  # nulls become NaN

# Use only interior edges; np.digitize returns 0…n_slices-1 directly
null_mask = spot_stats_df[z_col].is_null().to_numpy()
layer_arr = np.digitize(np.nan_to_num(z_arr, nan=bins[0] - 1), bins[1:-1]).astype("int32")

# Build series then scatter nulls back at the original null positions
z_layer_series = pl.Series("Position Z Layer", layer_arr, dtype=pl.Int32).scatter(
    np.where(null_mask)[0].tolist(), None
)

# Insert immediately after z_col
z_idx = spot_stats_df.columns.index(z_col)
col_order = (
    spot_stats_df.columns[: z_idx + 1]
    + ["Position Z Layer"]
    + [c for c in spot_stats_df.columns[z_idx + 1 :] if c != "Position Z Layer"]
)
spot_stats_df = spot_stats_df.with_columns(z_layer_series).select(col_order)

# Validation: per-layer spot count, Z min/max, and check against bin edges
layer_col = "Position Z Layer"
summary = (
    spot_stats_df
    .group_by(layer_col)
    .agg(
        pl.len().alias("n_spots"),
        pl.col(z_col).min().alias("z_min"),
        pl.col(z_col).max().alias("z_max"),
    )
    .sort(layer_col)
    .with_columns([
        pl.Series("bin_edge_lo", bins[:-1]),
        pl.Series("bin_edge_hi", bins[1:]),
    ])
    .with_columns([
        (pl.col("z_min") >= pl.col("bin_edge_lo")).alias("lo_ok"),
        (pl.col("z_max") <= pl.col("bin_edge_hi")).alias("hi_ok"),
    ])
)
summary

Position Z Layer,n_spots,z_min,z_max,bin_edge_lo,bin_edge_hi,lo_ok,hi_ok
i32,u64,f64,f64,f64,f64,bool,bool
0,8521,0.75,0.75,0.75,1.137,true,true
1,22602,1.50052,2.898567,1.137,2.901,true,true
2,39933,2.901183,4.447547,2.901,4.449,true,true
3,38236,4.449717,5.942409,4.449,5.943,true,true
4,31732,5.94368,7.454849,5.943,7.455,true,true
5,21066,7.457509,8.997123,7.455,9.363,true,true
6,12917,9.75,9.75,9.363,9.75,true,true


### Fix mismatch Imaris's and raw CZI/Cellpose's coordinate systems
Imaris's often infer a slightly different image size and origin than the raw CZI image, resulting in a systematic offset between the puncta coordinates and the Cellpose segmentation.

We need to do some work to correct for this offset to properly align the detected puncta spots to the Cellpose segmentation masks.

There are 2 options:
1. *(Recommended)* Go to [Paths Setup](#Paths-Setup) and set `IMARIS_IMS_FILE_IMAGE_INFO` with the correct values seen in the **Imaris Viewer** software. Or
2. Set the values in `IMARIS_IMS_FILE_IMAGE_INFO` to `None` and the code will fallback to using the Distance to Image Border inference method to estimate the image extent and correct the coordinates accordingly. This may or may not work depending heavily on the Imaris dataset. **NOT RECOMMENDED**, please use option 1 if possible.

In [ ]:
# Resolve Imaris-vs-CZI XY scaling for spot -> cellpose pixel alignment.
#
# Switch:
# 1) IMARIS_IMS_FILE_IMAGE_INFO[*].microns_per_pixel:
#    - if None, inherit RAW_CZI metadata (size_metadata[*]['micron_per_pixel'])
# 2) IMARIS_IMS_FILE_IMAGE_INFO[*].pixels:
#    - if None, keep current behavior and infer extent from spots + border-distance
#    - if set, compare CZI vs IMS metadata to compute the correction ratio for spot_scale

# "Distance to Image Border XY" = min(x, W−x, y, H−y)
# -> max(pos_i + dist_i) over all spots approximates the image extent used by Imaris.
dist_xy_col = next(c for c in spot_stats_df.columns if "Distance to Image Border XY" in c and "XYZ" not in c)
print(f"Border-distance columns found:\n  XY  -> {dist_xy_col}")

x_col = axes_colnames["X"]
y_col = axes_colnames["Y"]

# Fallback inferred extents (µm)
inferred_W = (spot_stats_df[x_col] + spot_stats_df[dist_xy_col]).max()
inferred_H = (spot_stats_df[y_col] + spot_stats_df[dist_xy_col]).max()

if inferred_W is None or inferred_H is None:
    raise ValueError("Could not infer image XY extent from spot positions and border-distance column")

inferred_W_val = float(inferred_W) # type: ignore
inferred_H_val = float(inferred_H) # type: ignore

print(f"\nInferred image dimensions (from spots + border distances):")
print(f"  Width  (X): {inferred_W_val:.4f} µm")
print(f"  Height (Y): {inferred_H_val:.4f} µm")

print(f"\nCZI metadata dimensions:")
print(f"  Width  (X): {size_metadata['X']['microns']:.4f} µm")
print(f"  Height (Y): {size_metadata['Y']['microns']:.4f} µm")

# cp_mask pixel dimensions (W_cp, H_cp)
W_cp_px, H_cp_px = cp_masks.shape[1], cp_masks.shape[0]
print(f"\nCellpose mask size: {W_cp_px} x {H_cp_px} px")


def _resolve_axis_scale(axis: str, inferred_extent_um: float) -> tuple[float, float, float, float | None, float]:
    """
    Returns:
        - axis_extent_um_used
        - spot_scale_axis (µm per cp_mask pixel)
        - correction_ratio_vs_czi
        - ims_pixels (or None)
        - ims_mpp
    """
    czi_px = float(size_metadata[axis]["pixels"])
    czi_mpp = float(size_metadata[axis]["micron_per_pixel"])
    czi_extent_um = float(size_metadata[axis]["microns"])

    ims_axis_info = IMARIS_IMS_FILE_IMAGE_INFO[axis]
    ims_pixels = ims_axis_info["pixels"]
    ims_mpp = float(ims_axis_info["microns_per_pixel"] if ims_axis_info["microns_per_pixel"] is not None else czi_mpp)

    if ims_pixels is None:
        # Fallback to use Distance to Image Border inference for axis extent
        axis_extent_um_used = float(inferred_extent_um)
    else:
        # Compare CZI and IMS metadata directly when IMS pixel size is explicitly provided
        pixel_ratio = float(ims_pixels) / czi_px
        mpp_ratio = ims_mpp / czi_mpp
        correction_ratio = pixel_ratio * mpp_ratio
        axis_extent_um_used = czi_extent_um * correction_ratio

    correction_ratio_vs_czi = axis_extent_um_used / czi_extent_um
    cp_axis_px = float(W_cp_px if axis == "X" else H_cp_px)
    spot_scale_axis = axis_extent_um_used / cp_axis_px
    return axis_extent_um_used, spot_scale_axis, correction_ratio_vs_czi, ims_pixels, ims_mpp


x_extent_um, spot_scale_x, corr_x, ims_x_pixels, ims_x_mpp = _resolve_axis_scale("X", inferred_W_val)
y_extent_um, spot_scale_y, corr_y, ims_y_pixels, ims_y_mpp = _resolve_axis_scale("Y", inferred_H_val)

print("\nEffective IMS axis settings (after None fallbacks):")
print(f"  X: pixels={ims_x_pixels}, microns_per_pixel={ims_x_mpp:.6f}, extent_used={x_extent_um:.4f} µm")
print(f"  Y: pixels={ims_y_pixels}, microns_per_pixel={ims_y_mpp:.6f}, extent_used={y_extent_um:.4f} µm")

print("\nCZI -> IMS correction ratios used for spot scaling:")
print(f"  X ratio: {corr_x:.6f}")
print(f"  Y ratio: {corr_y:.6f}")

print(f"\nCorrected spot -> pixel scale factors:")
print(f"  spot_scale_x: {spot_scale_x:.6f} µm/px  (was cp_scale_x: {cp_scale_x:.6f})")
print(f"  spot_scale_y: {spot_scale_y:.6f} µm/px  (was cp_scale_y: {cp_scale_y:.6f})")

Border-distance columns found:
  XY  -> Distance to Image Border XY Img=1 [µm]

Inferred image dimensions (from spots + border distances):
  Width  (X): 1958.4100 µm
  Height (Y): 593.1522 µm

CZI metadata dimensions:
  Width  (X): 1930.3747 µm
  Height (Y): 593.9814 µm

Cellpose mask size: 9315 x 2864 px

Effective IMS axis settings (after None fallbacks):
  X: pixels=37824, microns_per_pixel=0.051790, extent_used=1958.9111 µm
  Y: pixels=11454, microns_per_pixel=0.051790, extent_used=593.2045 µm

CZI -> IMS correction ratios used for spot scaling:
  X ratio: 1.014783
  Y ratio: 0.998692

Corrected spot -> pixel scale factors:
  spot_scale_x: 0.210296 µm/px  (was cp_scale_x: 0.207233)
  spot_scale_y: 0.207124 µm/px  (was cp_scale_y: 0.207396)


In [ ]:
# Transform spot XY positions (µm) -> cp_mask pixel coordinates
# Uses resolved spot_scale_x/y from IMARIS_IMS_FILE_IMAGE_INFO + CZI comparison logic.
x_col = axes_colnames['X']
y_col = axes_colnames['Y']

spot_stats_df = spot_stats_df.with_columns([
    (pl.col(x_col) / spot_scale_x).alias("cp_pixel_x"),
    (pl.col(y_col) / spot_scale_y).alias("cp_pixel_y"),
])

# Random sample of spots for overlay
rng = np.random.default_rng(42)
n_sample = min(200_000, len(spot_stats_df))
sample_idx = rng.choice(len(spot_stats_df), size=n_sample, replace=False)
sample_df = spot_stats_df[sample_idx]

sx = sample_df["cp_pixel_x"].drop_nulls().to_numpy()
sy = sample_df["cp_pixel_y"].drop_nulls().to_numpy()

# Plot
H_cp, W_cp = cp_masks.shape  # (Y, X)

fig2 = go.Figure()
fig2.add_trace(go.Image(z=export_img, name="Cellpose overlay"))
fig2.add_trace(go.Scatter(
    x=sx * scale,
    y=sy * scale,
    mode="markers",
    marker=dict(color="red", size=4, opacity=0.6),
    name=f"Spots (n={n_sample:,})",
))

fig2.update_layout(
    title="Spot XY aligned to Cellpose pixel space",
    paper_bgcolor="black",
    plot_bgcolor="black",
    margin=dict(l=60, r=20, t=40, b=60),
    width=eW,
    height=eH,
    xaxis=dict(
        range=[0, eW],
        showgrid=False,
        title=dict(text="X (px -> cp_mask)", font=dict(color="white")),
        tickfont=dict(color="white"),
    ),
    yaxis=dict(
        range=[eH, 0], # flip so origin is top-left, matching image convention
        showgrid=False,
        title=dict(text="Y (px -> cp_mask)", font=dict(color="white")),
        tickfont=dict(color="white"),
    ),
    legend=dict(font=dict(color="white")),
)

imgoutpath = os.path.normpath(os.path.join(OUTPUT_DIR, "spot_alignment_qc.webp"))
fig2.write_image(imgoutpath, format="webp")
print(f"Saved to: {imgoutpath}")

Saved to: D:\JOBS\WashU_Neuroscience\RNAScope\RNAScope_DATA\TEST_ACxDev1_CBA-CAJ_P18_Male_LACx_ONLY\data\spot_alignment_qc.webp


### Spots to Cells Assignment

In [15]:
# For each spot, in the cellpose pixel space, check if it falls within a cell (cp_masks > 0) or outside (cp_masks == 0)
# If within a cell, assign the spot the corresponding cell label from cp_masks; if outside mask, assign -1; if out of bounds or null coords, assign null
def assign_cell_label(row):
    x, y = row["cp_pixel_x"], row["cp_pixel_y"]
    if x is None or y is None:
        return None
    xi, yi = int(x), int(y)
    H_cp, W_cp = cp_masks.shape
    if xi < 0 or xi >= W_cp or yi < 0 or yi >= H_cp:
        return None
    label = cp_masks[yi, xi]
    return int(label) if label > 0 else -1

# Vectorized lookup (much faster than row-wise apply for large DataFrames)
H_cp, W_cp = cp_masks.shape

px = spot_stats_df["cp_pixel_x"].to_numpy()
py = spot_stats_df["cp_pixel_y"].to_numpy()

null_mask = (
    spot_stats_df["cp_pixel_x"].is_null().to_numpy()
    | spot_stats_df["cp_pixel_y"].is_null().to_numpy()
)

# Out-of-bounds: coords exist but fall outside the cp_mask extent
oob_mask = (
    ~null_mask
    & ((px < 0) | (px >= W_cp) | (py < 0) | (py >= H_cp))
)

valid = ~null_mask & ~oob_mask

# Lookup cp_masks only for in-bounds spots; start everything as -1
labels = np.full(len(spot_stats_df), -1, dtype="int32")
xi = px[valid].astype(int)
yi = py[valid].astype(int)
mask_vals = cp_masks[yi, xi].astype("int32")
labels[valid] = np.where(mask_vals > 0, mask_vals, -1)

# Scatter nulls back for null-coord and out-of-bounds rows
null_indices = np.where(null_mask | oob_mask)[0].tolist()
cell_label_series = pl.Series("cell_label", labels, dtype=pl.Int32).scatter(null_indices, None)
spot_stats_df = spot_stats_df.with_columns(cell_label_series)

# Quick summary
n_in   = (spot_stats_df["cell_label"] > 0).sum()
n_out  = (spot_stats_df["cell_label"] == -1).sum()
n_null = spot_stats_df["cell_label"].is_null().sum()
print(f"Spots inside a cell        : {n_in:,}")
print(f"Spots outside cells        : {n_out:,}")
print(f"Spots null (oob/null coord) : {n_null:,}")


Spots inside a cell        : 85,310
Spots outside cells        : 89,697
Spots null (oob/null coord) : 0


## Exporting Vizgen-Compatible Files

In [16]:
# Get the last modified timestamp of the cellpose_npy file and use it as the prefix for cell_id
# NOTE! that the timestamp WILL change when you re-run cellpose!!!

cellpose_timestamp = int(os.path.getmtime(CELLPOSE_NPY))

# Get the max cell label from cp_masks to determine padding width
max_cell_label = cp_masks.max()
pad_width = len(str(max_cell_label))

def format_cell_label(prefix: int, cell_label: int, pad: int) -> int:
	if cell_label is None or cell_label < 0:
		return -1
	else:
		return int(f"{prefix}{cell_label:0{pad}d}")


In [17]:
def cp_masks_cell_stats(masks: np.ndarray, scale_x: float, scale_y: float) -> pl.DataFrame:
    """
    Compute per-cell volume (µm²) and XY bounding box (µm) directly from a cellpose mask array.

    np.argwhere gives (row=y_px, col=x_px) for every labeled pixel in one
    vectorised pass. A Polars groupby then computes all aggregates without any Python loop.

    Returns a DataFrame with columns:
        cell_label, volume, min_x, max_x, min_y, max_y
    """
    yx = np.argwhere(masks > 0)        # shape (N, 2)
    labels = masks[yx[:, 0], yx[:, 1]] # cell label per pixel

    return (
        pl.DataFrame({
            "cell_label": pl.Series(labels.astype("int32")),
            "x_px":       pl.Series(yx[:, 1].astype("float64")),  # col -> X
            "y_px":       pl.Series(yx[:, 0].astype("float64")),  # row -> Y
        })
        .group_by("cell_label")
        .agg(
            volume=pl.len() * scale_x * scale_y,
            center_x=pl.col("x_px").mean() * scale_x,
            center_y=pl.col("y_px").mean() * scale_y,
            min_x=pl.col("x_px").min() * scale_x,
            max_x=pl.col("x_px").max() * scale_x,
            min_y=pl.col("y_px").min() * scale_y,
            max_y=pl.col("y_px").max() * scale_y,
        )
    )

cp_cell_stats_df = cp_masks_cell_stats(cp_masks, spot_scale_x, spot_scale_y)

cp_cell_stats_df.head()

cell_label,volume,center_x,center_y,min_x,max_x,min_y,max_y
i32,f64,f64,f64,f64,f64,f64,f64
530,228.807727,1380.96768,95.543766,1371.553183,1391.110749,88.235028,103.355115
2370,49.960492,1634.627391,409.698899,1630.428063,1639.260512,405.549728,414.248956
2230,218.702379,860.799077,389.369557,852.962237,869.155061,380.487666,398.093247
2885,227.805903,1418.173766,500.192427,1407.303573,1428.122917,493.163383,507.454972
1188,165.649302,414.387112,208.10482,406.292663,422.485487,201.532117,215.202333


In [18]:
# detected_transcripts
# Match, update, and rename headers to match old format:
# ,barcode_id,global_x,global_y,global_z,x,y,fov,gene,transcript_id,cell_id
# All spots here are Shank3, ENSMUSG00000022623, always channel 2 (Ch2)
# Col1: _ --> -1 (first column has no header and all values are -1)
# Col2: barcode_id -> "Ch2"
# Col3-5: global_x/y/z -> x/y/z (already in µm)
# Col6-8: x,y,fov -> -1
# Col9: gene -> "Shank3"
# Col10: transcript_id -> ENSMUSG00000022623
# Col11: cell_id -> cell_label (from cp_masks lookup), if null then set to -1 for compatibility with downstream code that expects an int

detected_transcripts_df = spot_stats_df.select([
    pl.lit(-1).alias(""),
    pl.lit("Ch2").alias("barcode_id"),
	pl.col(axes_colnames['X']).alias("global_x"),
	pl.col(axes_colnames['Y']).alias("global_y"),
	pl.col("Position Z Layer").alias("global_z"),
	pl.lit(-1).alias("x"),
	pl.lit(-1).alias("y"),
	pl.lit(-1).alias("fov"),
	pl.lit("Shank3").alias("gene"),
	pl.lit("ENSMUSG00000022623").alias("transcript_id"),
    # use the format_cell_label function to combine the cellpose timestamp and cell label into a single integer ID, with nulls as -1
	pl.col("cell_label").map_elements(lambda label: format_cell_label(cellpose_timestamp, label, pad_width), return_dtype=pl.Int64, skip_nulls=False).alias("cell_id"),
])

csvoutpath = os.path.normpath(os.path.join(OUTPUT_DIR, "detected_transcripts.csv"))
detected_transcripts_df.write_csv(csvoutpath)
print(f"Saved to: {csvoutpath}")

Saved to: D:\JOBS\WashU_Neuroscience\RNAScope\RNAScope_DATA\TEST_ACxDev1_CBA-CAJ_P18_Male_LACx_ONLY\data\detected_transcripts.csv


In [19]:
# cell_metadata
# EntityID,fov,volume,center_x,center_y,min_x,min_y,max_x,max_y,anisotropy,transcript_count,perimeter_area_ratio,solidity
# Group by cell_label, then calculate:
# EntityID -> cell_id (same formatting as above, using the cellpose timestamp and cell label)
# fov -> -1
# volume -> count of pixels in cp_masks for this cell_label * (spot_scale_x * spot_scale_y) to convert to µm²
# center_x/y -> mean of cp_pixel_x/y for spots in this cell, converted to µm using spot_scale_x/y
# min_x/y and max_x/y -> inferred from the cp_masks pixel belonging to this cell_label, converted to µm using spot_scale_x/y
# anisotropy -> -1
# transcript_count -> count of spots in this cell
# perimeter_area_ratio -> -1
# solidity -> -1

cell_metadata_df = (
	spot_stats_df
	.filter(pl.col("cell_label") > 0)  # only consider spots that are inside cells
	.group_by("cell_label")
	.agg(
		transcript_count=pl.len(),
	)
	.join(cp_cell_stats_df, on="cell_label", how="left")  # brings in volume, center_x, center_y, min_x, max_x, min_y, max_y
	.with_columns(
		EntityID=pl.col("cell_label").map_elements(lambda label: format_cell_label(cellpose_timestamp, label, pad_width), return_dtype=pl.Int64),
		fov=pl.lit(-1),
		anisotropy=pl.lit(-1),
		perimeter_area_ratio=pl.lit(-1),
		solidity=pl.lit(-1)
	)
	.select([
		"EntityID", "fov", "volume", "center_x", "center_y",
		"min_x", "min_y", "max_x", "max_y",
		"anisotropy", "transcript_count", "perimeter_area_ratio", "solidity"
	])
)

csvoutpath = os.path.normpath(os.path.join(OUTPUT_DIR, "cell_metadata.csv"))
cell_metadata_df.write_csv(csvoutpath)
print(f"Saved to: {csvoutpath}")


Saved to: D:\JOBS\WashU_Neuroscience\RNAScope\RNAScope_DATA\TEST_ACxDev1_CBA-CAJ_P18_Male_LACx_ONLY\data\cell_metadata.csv


In [20]:
# top 10 cells with most transcripts
top_cells = cell_metadata_df.sort("transcript_count", descending=True).head(10)
print("Top 10 cells by transcript count:")
print(top_cells.select(["EntityID", "transcript_count"]))

Top 10 cells by transcript count:
shape: (10, 2)
┌────────────────┬──────────────────┐
│ EntityID       ┆ transcript_count │
│ ---            ┆ ---              │
│ i64            ┆ u64              │
╞════════════════╪══════════════════╡
│ 17794686062827 ┆ 104              │
│ 17794686061981 ┆ 95               │
│ 17794686061968 ┆ 94               │
│ 17794686062309 ┆ 94               │
│ 17794686061611 ┆ 93               │
│ 17794686063285 ┆ 91               │
│ 17794686060295 ┆ 89               │
│ 17794686061086 ┆ 89               │
│ 17794686060215 ┆ 87               │
│ 17794686061392 ┆ 86               │
└────────────────┴──────────────────┘


In [21]:
# pick a random cell from the top 100, plot the cell mask and overlay the spots assigned to that cell to visually check the assignments
top_100 = cell_metadata_df.sort("transcript_count", descending=True).head(100)
random_cell = top_100.sample(1, seed=42).select("EntityID").item()
print(f"Randomly selected cell for QC: {random_cell}")

# Get the cp_mask label corresponding to this EntityID by reversing the format_cell_label functions
def reverse_format_cell_label(entity_id: int, prefix: int, pad: int) -> int:
	s = str(entity_id)
	prefix_str = str(prefix)
	if not s.startswith(prefix_str):
		raise ValueError(f"EntityID {entity_id} does not start with expected prefix {prefix}")
	label_str = s[len(prefix_str):]
	if len(label_str) != pad:
		raise ValueError(f"EntityID {entity_id} has label part with unexpected length (expected {pad} digits, got {len(label_str)})")
	return int(label_str)
cell_label = reverse_format_cell_label(random_cell, cellpose_timestamp, pad_width)
print(f"Corresponding cell_label in cp_masks: {cell_label}")


Randomly selected cell for QC: 17794686060172
Corresponding cell_label in cp_masks: 172


In [22]:
# Crop region around the selected cell
cell_row = cell_metadata_df.filter(pl.col("EntityID") == random_cell).row(0, named=True)

pad_px = 80  # padding around bounding box in cp_mask pixels
H_cp_px, W_cp_px = cp_masks.shape

x0 = max(0, int(cell_row["min_x"] / spot_scale_x) - pad_px)
x1 = min(W_cp_px, int(cell_row["max_x"] / spot_scale_x) + pad_px)
y0 = max(0, int(cell_row["min_y"] / spot_scale_y) - pad_px)
y1 = min(H_cp_px, int(cell_row["max_y"] / spot_scale_y) + pad_px)

# Crop origin in global µm space
x0_um = x0 * spot_scale_x
y0_um = y0 * spot_scale_y

crop_img      = tif_img[y0:y1, x0:x1]
crop_mask     = cp_masks[y0:y1, x0:x1]
crop_outlines = cp_outlines[y0:y1, x0:x1]

# Build display image
if crop_img.ndim == 2:
    crop_display = np.stack([crop_img] * 3, axis=-1)
elif crop_img.shape[-1] == 4:
    crop_display = crop_img[..., :3].copy()
else:
    crop_display = crop_img.copy()
crop_display = crop_display.astype(np.uint8)

cell_px = (crop_mask == cell_label)
crop_display[cell_px] = np.clip(
    crop_display[cell_px].astype(np.int32) + np.array([0, 60, 0]), 0, 255
).astype(np.uint8)
crop_display[crop_outlines > 0] = [255, 255, 255]

from scipy.ndimage import binary_dilation
sel_outline = binary_dilation(cell_px, iterations=2) & ~cell_px
crop_display[sel_outline] = [0, 255, 0]

# Spots in global µm
cell_spots = spot_stats_df.filter(pl.col("cell_label") == cell_label)
spot_x_um = cell_spots[axes_colnames['X']].to_numpy()   # already in global µm
spot_y_um = cell_spots[axes_colnames['Y']].to_numpy()

cH, cW = crop_display.shape[:2]
cW_um = cW * spot_scale_x
cH_um = cH * spot_scale_y

# Margins: figure size = inner plot + margins
_ml, _mr, _mt, _mb = 70, 20, 70, 60

# Plot
fig_qc = go.Figure()
# x0/y0 anchor the image at the crop's global µm origin; dx/dy give physical pixel size
fig_qc.add_trace(go.Image(
    z=crop_display,
    x0=x0_um, dx=spot_scale_x,
    y0=y0_um, dy=spot_scale_y,
    name="Cell region",
))
fig_qc.add_trace(go.Scatter(
    x=spot_x_um,
    y=spot_y_um,
    mode="markers",
    marker=dict(color="red", size=8, opacity=0.85, line=dict(color="white", width=0.5)),
    name=f"Spots in cell (n={len(cell_spots):,})",
))
fig_qc.update_layout(
    title=dict(
        text=f"QC — EntityID {random_cell}<br><sup>mask label {cell_label} · {len(cell_spots)} transcripts</sup>",
        font=dict(color="white"),
        x=0.5,
        xanchor="center",
    ),
    paper_bgcolor="black",
    plot_bgcolor="black",
    margin=dict(l=_ml, r=_mr, t=_mt, b=_mb, pad=0),
    width=cW + _ml + _mr,
    height=cH + _mt + _mb,
    xaxis=dict(
        range=[x0_um, x0_um + cW_um], showgrid=False,
        title=dict(text="X (µm)", font=dict(color="white")),
        tickfont=dict(color="white"),
        ticksuffix=" µm",
    ),
    yaxis=dict(
        range=[y0_um + cH_um, y0_um], showgrid=False,   # top-left origin, flipped
        title=dict(text="Y (µm)", font=dict(color="white")),
        tickfont=dict(color="white"),
        ticksuffix=" µm",
    ),
    legend=dict(font=dict(color="white")),
)
fig_qc.show()

imgoutpath = os.path.normpath(os.path.join(OUTPUT_DIR, f"qc_cell_{random_cell}.webp"))
fig_qc.write_image(imgoutpath, format="webp", scale=2)
print(f"Saved to: {imgoutpath}")

Saved to: D:\JOBS\WashU_Neuroscience\RNAScope\RNAScope_DATA\TEST_ACxDev1_CBA-CAJ_P18_Male_LACx_ONLY\data\qc_cell_17794686060172.webp


In [23]:
# cell_by_gene
# Rows = cells, columns = genes, values = transcript counts per cell per gene
cell_by_gene_df = (
    detected_transcripts_df
    .filter(pl.col("cell_id") > 0)   # exclude spots not assigned to a cell (-1)
    .group_by(["cell_id", "gene"])
    .agg(pl.len().alias("count"))
    .pivot(on="gene", index="cell_id", values="count", aggregate_function="sum")
    .fill_null(0)
    .rename({"cell_id": "cell"})
    .sort("cell")
)

csvoutpath = os.path.normpath(os.path.join(OUTPUT_DIR, "cell_by_gene.csv"))
cell_by_gene_df.write_csv(csvoutpath)
print(f"Saved to: {csvoutpath}")

cell_by_gene_df

Saved to: D:\JOBS\WashU_Neuroscience\RNAScope\RNAScope_DATA\TEST_ACxDev1_CBA-CAJ_P18_Male_LACx_ONLY\data\cell_by_gene.csv


cell,Shank3
i64,u64
17794686060001,2
17794686060002,4
17794686060003,1
17794686060004,24
17794686060005,18
…,…
17794686063401,1
17794686063402,6
17794686063403,1
